# Tournament Manager — Alembic Migrations

This notebook demonstrates how to:
1. Check the current migration state
2. Apply **Migration 1** — Initial schema creation
3. Apply **Migration 2** — Added `last_update` attribute
4. Verify the resulting schema
5. Downgrade back to base (cleanup)

> **Prerequisites:** `pip install -r requirements.txt` must have been run first.
>
> The notebook uses `APP_ENV=test` (SQLite) so no external database is needed.

## 0. Setup — point Alembic at the test SQLite database

In [ ]:
import os, sys, subprocess

# Project root must be on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)  # alembic.ini must be in cwd

os.environ["APP_ENV"] = "test"
os.environ["DATABASE_URL_TEST"] = "sqlite:///./migrations_demo.db"

def alembic(cmd: str):
    """Run an alembic CLI command and print its output."""
    result = subprocess.run(
        ["alembic"] + cmd.split(),
        capture_output=True, text=True
    )
    output = (result.stdout + result.stderr).strip()
    print(output if output else "(no output)")
    if result.returncode != 0:
        raise RuntimeError(f"alembic {cmd!r} failed")

print("✅ Setup complete — working dir:", os.getcwd())

## 1. Check initial migration state

In [ ]:
print("── Migration history ─────────────────────────")
alembic("history --verbose")

In [ ]:
print("── Current revision (should be empty / None) ──")
alembic("current")

## 2. Apply Migration 1 — Initial schema creation

In [ ]:
alembic("upgrade 001_initial_schema")
print("✅ Migration 1 applied")

In [ ]:
# Verify tables exist and columns are correct
from sqlalchemy import create_engine, inspect

engine = create_engine("sqlite:///./migrations_demo.db")
insp = inspect(engine)

expected_tables = [
    "fighter", "medical_record", "tournament",
    "tatami", "category", "fighter_category_registration", "match"
]

actual_tables = insp.get_table_names()
print("Tables in DB:", actual_tables)

for t in expected_tables:
    assert t in actual_tables, f"Table '{t}' missing!"

print("\n✅ All 7 tables present after Migration 1")

In [ ]:
# Show fighter columns after Migration 1 (no last_update yet)
fighter_cols = [c["name"] for c in insp.get_columns("fighter")]
print("fighter columns (Migration 1):", fighter_cols)
assert "last_update" not in fighter_cols, "last_update should NOT exist yet"
print("✅ last_update column is absent (as expected)")

## 3. Apply Migration 2 — Added last_update attribute

In [ ]:
alembic("upgrade 002_add_last_update")
print("✅ Migration 2 applied")

In [ ]:
# Reconnect inspector to pick up schema changes
engine.dispose()
engine = create_engine("sqlite:///./migrations_demo.db")
insp = inspect(engine)

tables_with_last_update = [
    "fighter", "medical_record", "tournament", "tatami",
    "category", "fighter_category_registration", "match"
]

for t in tables_with_last_update:
    cols = [c["name"] for c in insp.get_columns(t)]
    assert "last_update" in cols, f"last_update missing in table '{t}'"
    print(f"  ✅  {t}: last_update present")

print("\n✅ Migration 2 verified — last_update added to all 7 tables")

In [ ]:
print("── Final revision ─────────────────────────────")
alembic("current")

## 4. Downgrade back to base (cleanup)

In [ ]:
alembic("downgrade base")
print("✅ Downgraded to base — all tables dropped")

In [ ]:
engine.dispose()
engine = create_engine("sqlite:///./migrations_demo.db")
insp = inspect(engine)
remaining = insp.get_table_names()

# Only alembic_version table may remain
domain_tables = [t for t in remaining if t != "alembic_version"]
assert domain_tables == [], f"Expected no domain tables, found: {domain_tables}"
print("✅ All domain tables removed. Remaining:", remaining)

## 5. Upgrade back to head (ready for use)

In [ ]:
alembic("upgrade head")
print("✅ Schema at latest revision (head)")
alembic("current")

In [ ]:
# Cleanup demo file
import os
if os.path.exists("migrations_demo.db"):
    os.remove("migrations_demo.db")
    print("✅ migrations_demo.db removed")

---
## ✅ All migration tests passed!

In [ ]:
print("=" * 50)
print("ALL MIGRATION TESTS PASSED SUCCESSFULLY ✅")
print("=" * 50)